In [ ]:
# To pull User data from Replay files, and then pull User data from PS.com

# Getting user data
------------

In [ ]:
@dataclass 
class User:
    username: str
    userid: str
    registertime: int
    group: int
    ratings: dict

# ===========================
def get_user(user_id: str, is_clean = False):
    '''
    Return a `User` object pulled from PS using `user_id`, 'cleaning' the string first.
    '''

    # clean by deleting anything not in {A-Z,a-z,0-9}, and then |-> lowercase
    if not is_clean:
        clean_user = lambda s : re.sub("[^A-Za-z0-9]", "", s).lower()
        user_id = clean_user(user_id)
    else : pass  

    url = urljoin(USER_BASE_URL, f'{user_id}.json')
    response = requests.get(url, timeout=30)
    
    try:
        data = response.json()
    except json.JSONDecodeError as e:
        logger.error(f"failed to parse user {user_id}: {e}")
        return None

    # ===========================
    # 'Unregistered' users are treated much like nonexistant users, but the former can 
    # have ratings. Hence, we suppress `404` errors and instead check whether a 
    # User's `ratings` field is empty or not.
    
    if data.get("ratings", {}) == {} :
        return None
    else :
        return User(
            username = data.get("username", ""),
            userid = data.get("userid", ""),
            registertime = data.get("registertime", 0),
            group = data.get("group", 0),
            ratings = data.get("ratings", dict())
        )

In [ ]:
# scrape raw usernames from replay files
# Note: for now, it seems simplest to keep the .txt of names in the `data` directory
# as `users_raw` and keep all actual user-data .json in the /users directory

REPLAY_DIR = Path('./replays/gen9-randombattle')

user_list = []

for item in REPLAY_DIR.iterdir():
    if (item.is_file() and item.name[0]!='.'): # skipping hidden files
        try:
            with open(REPLAY_DIR/f'{item.name}', 'r') as file:
                x = json.load(file)
                for user in x.get('players',[]):
                    user_list.append(player)
        except UnicodeDecodeError as e:
            print(f"{item.name}")
    else : 
        print(f'{item.name}')
        continue

# can change to 'a' mode later
with open('./users_raw.txt', 'w') as file: 
    for user in user_list : file.write(user + '\n')

In [ ]:
# For pulling actual user data from PS
ensure_dir('./users')
USER_DIR = Path('./users')

user_errors = [] # anyone we fail(ed) to get data for 
user_errors_fp = 'user_errors.txt' # where we'll write errors

# grab names from file into array 
with open('users_raw.txt', 'r') as file:
    users_raw = file.readlines()
users_raw = list(map(lambda s : s[:-1], users_raw)) # strip tailing '\n' from each entry

for username in users_raw:
    # Pull User (object) from PS
    user = get_user(username)
    
    if user is not None :
        with open(USER_DIR/f'{user.userid}.json', 'w') as file:
            json.dump(asdict(user), file)
    else : user_errors.append(username)

if user_errors == []:
    print(f"Successfully pulled data for all {len(users_raw)} users.")
else : 
    print(f"Failed to get data for {len(user_errors)} users; writing to {user_errors_fp}.")
    with open(user_errors_fp, 'a') as file:
        for user in user_errors:
            file.write(user)